# Attack 1 — Word Deletion

Reproduction of the word-deletion attack from Gloaguen et al. (ICLR 2026), Figure 3 (left).
Each word is deleted independently with probability equal to the deletion ratio
(MarkLLM `WordDeletion`). Ratios: 0.1, 0.2, 0.3, 0.4, 0.5, as in the official
`scripts/run_robustness_eval.py`.

Detection: local detector on retokenized text (`src/detection.py`), threshold p < 0.01.
Runs locally on Mac (CPU).

## 1. Setup

In [1]:
import random
import sys
from importlib.metadata import version
from pathlib import Path

import numpy as np


def find_project_root(start: Path) -> Path:
    """Walk up the folders until the official ETH clone is found."""
    for folder in [start, *start.parents]:
        if (folder / "diffusion-lm-watermark").exists():
            return folder
    raise FileNotFoundError("Project root not found.")


ROOT = find_project_root(Path.cwd().resolve())
sys.path.insert(0, str(ROOT / "src"))
from detection import ALPHA, detect, load_detector, load_jsonl, save_jsonl

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

WM_PATH = ROOT / "results" / "Cleaned Data" / "watermarked_600.jsonl"
OUT_DIR = ROOT / "results" / "attacks" / "word_deletion"

wm = load_jsonl(WM_PATH)
tokenizer, detector = load_detector(ROOT)

print("Project root:", ROOT)
print("Watermarked records:", len(wm))
for pkg in ["torch", "transformers", "markllm", "nltk"]:
    print(f"{pkg}: {version(pkg)}")

/Users/paulmicheli/Desktop/Unine/MA1/Trustworthy/Trustworhty_AI_Watermarking/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Could not reach the Hub ([Errno 8] nodename nor servname provided, or not known). Using cached commit hash for 'GSAI-ML/LLaDA-8B-Instruct'.


Project root: /Users/paulmicheli/Desktop/Unine/MA1/Trustworthy/Trustworhty_AI_Watermarking
Watermarked records: 600
torch: 2.15.0
transformers: 5.22.1
markllm: 0.1.5
nltk: 3.10.4


## 2. Control: reproduce the baseline

We rerun the shared detector on the 600 original watermarked texts. It must give
583/600 detections and the same p-values as `results/detection_results.jsonl`
(notebook 03). The extended baseline (green fraction, scored tokens) is saved once
and reused by all attack notebooks.

In [2]:
from collections import defaultdict

# Run the shared detector on the original texts
baseline = []
for i, row in enumerate(wm):
    res = detect(row["completion"], tokenizer, detector)
    baseline.append({
        "idx": i,
        "dataset": row["dataset_name"].split("_batch_")[0],
        **res,
    })
    if (i + 1) % 100 == 0:
        print(f"Processed {i + 1}/600")

# Check 1: same number of detections as notebook 03
n_detected = sum(r["detected"] for r in baseline)
print(f"\nDetected: {n_detected}/600 (expected 583)")

# Check 2: same p-values as the saved detection results
saved = [r for r in load_jsonl(ROOT / "results" / "detection_results.jsonl")
         if r["condition"] == "watermarked"]
max_diff = max(abs(a["p_value"] - b["p_value"]) for a, b in zip(baseline, saved))
print(f"Max p-value difference vs notebook 03: {max_diff:.2e} (expected 0)")

# Per-dataset summary
by_ds = defaultdict(list)
for r in baseline:
    by_ds[r["dataset"]].append(r)
print("\nDataset            TPR     mean green fraction   mean scored tokens")
for ds, rows in by_ds.items():
    tpr = sum(r["detected"] for r in rows) / len(rows)
    gf = np.mean([r["green_fraction"] for r in rows])
    ns = np.mean([r["n_scored"] for r in rows])
    print(f"{ds:18s} {tpr:6.1%}   {gf:19.3f}   {ns:18.1f}")

Processed 100/600
Processed 200/600
Processed 300/600
Processed 400/600
Processed 500/600
Processed 600/600

Detected: 583/600 (expected 583)
Max p-value difference vs notebook 03: 0.00e+00 (expected 0)

Dataset            TPR     mean green fraction   mean scored tokens
2-1_longform_qa     99.5%                 0.452                193.7
2-2_finance_qa      99.0%                 0.447                187.1
5-1_alpacafarm      93.0%                 0.431                178.1


In [3]:
BASELINE_PATH = ROOT / "results" / "attacks" / "baseline_original.jsonl"
save_jsonl(baseline, BASELINE_PATH)

Saved 600 records to /Users/paulmicheli/Desktop/Unine/MA1/Trustworthy/Trustworhty_AI_Watermarking/results/attacks/baseline_original.jsonl


## 3. Pilot: 5 texts

We apply MarkLLM `WordDeletion` (unchanged, as imported by the official robustness script)
to 5 texts at ratios 0.0, 0.1 and 0.3. Ratio 0.0 is an added control: it deletes nothing
but still applies the whitespace normalization done by `split()` / `' '.join()`.
The random seed is reset for every (ratio, text) pair so each result is reproducible.

In [4]:
from markllm.evaluation.tools.text_editor import WordDeletion

PILOT_IDX = [0, 1, 200, 201, 400]   # 2 Longform QA, 2 Finance QA, 1 AlpacaFarm
PILOT_RATIOS = [0.0, 0.1, 0.3]

for idx in PILOT_IDX:
    original = wm[idx]["completion"]
    base = baseline[idx]
    print("=" * 100)
    print(f"Text {idx} | {base['dataset']} | original: p={base['p_value']:.2e}  "
          f"green={base['green_fraction']:.3f}  scored={base['n_scored']}  tokens={base['n_tokens']}")
    print("ORIGINAL:", original[:300].replace("\n", "\\n"), "...")

    for ratio in PILOT_RATIOS:
        random.seed(SEED + idx)              # same seed for a given text, whatever the ratio
        attacked = WordDeletion(ratio=ratio).edit(original)
        res = detect(attacked, tokenizer, detector)
        n_words_orig, n_words_att = len(original.split()), len(attacked.split())
        print(f"\n  ratio {ratio:.1f} | words {n_words_orig} -> {n_words_att} | "
              f"p={res['p_value']:.2e}  green={res['green_fraction']:.3f}  "
              f"scored={res['n_scored']}  detected={res['detected']}")
        print("  ATTACKED:", attacked[:300], "...")

W1112 14:15:55.727000 45356 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Text 0 | 2-1_longform_qa | original: p=2.81e-15  green=0.480  scored=252  tokens=298
ORIGINAL: The collapse of the Soviet Union is when one of the largest superpowers, like a big super-strong robot that had been fighting together for many years, suddenly falls apart all by itself. Sometimes, a robot might fall apart because they don't work together anymore, or maybe they've gotten too far awa ...

  ratio 0.0 | words 250 -> 250 | p=4.66e-16  green=0.488  scored=250  detected=True
  ATTACKED: The collapse of the Soviet Union is when one of the largest superpowers, like a big super-strong robot that had been fighting together for many years, suddenly falls apart all by itself. Sometimes, a robot might fall apart because they don't work together anymore, or maybe they've gotten too far awa ...

  ratio 0.1 | words 250 -> 223 | p=6.86e-14  green=0.478  scored=230  detected=True
  ATTACKED: The of the Soviet Union is one the largest like a big super-strong robot that been fighting together 

## 4. Full run: 600 texts x 6 ratios

Ratios 0.1–0.5 reproduce the paper; ratio 0.0 is the whitespace-normalization control.
The seed depends only on the text index, so deletions are nested across ratios
(a word deleted at 0.1 is also deleted at 0.3). One output file per ratio.

In [5]:
RATIOS = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5]
MARKLLM_VERSION = version("markllm")

for ratio in RATIOS:
    out_path = OUT_DIR / f"word_deletion_r{ratio:.1f}.jsonl"
    records = []
    for idx, row in enumerate(wm):
        original = row["completion"]
        random.seed(SEED + idx)
        attacked = WordDeletion(ratio=ratio).edit(original)
        res = detect(attacked, tokenizer, detector)
        base = baseline[idx]
        records.append({
            "idx": idx,
            "dataset": base["dataset"],
            "attack": "word_deletion",
            "ratio": ratio,
            "seed": SEED + idx,
            "markllm_version": MARKLLM_VERSION,
            "prompt": row["prompt"],
            "original": original,
            "attacked": attacked,
            "n_words_original": len(original.split()),
            "n_words_attacked": len(attacked.split()),
            "original_p_value": base["p_value"],
            "original_detected": base["detected"],
            "original_green_fraction": base["green_fraction"],
            **res,
        })
    save_jsonl(records, out_path)
    n_det = sum(r["detected"] for r in records if r["detected"] is not None)
    print(f"ratio {ratio:.1f}: detected {n_det}/600")

Saved 600 records to /Users/paulmicheli/Desktop/Unine/MA1/Trustworthy/Trustworhty_AI_Watermarking/results/attacks/word_deletion/word_deletion_r0.0.jsonl
ratio 0.0: detected 585/600
Saved 600 records to /Users/paulmicheli/Desktop/Unine/MA1/Trustworthy/Trustworhty_AI_Watermarking/results/attacks/word_deletion/word_deletion_r0.1.jsonl
ratio 0.1: detected 569/600
Saved 600 records to /Users/paulmicheli/Desktop/Unine/MA1/Trustworthy/Trustworhty_AI_Watermarking/results/attacks/word_deletion/word_deletion_r0.2.jsonl
ratio 0.2: detected 547/600
Saved 600 records to /Users/paulmicheli/Desktop/Unine/MA1/Trustworthy/Trustworhty_AI_Watermarking/results/attacks/word_deletion/word_deletion_r0.3.jsonl
ratio 0.3: detected 522/600
Saved 600 records to /Users/paulmicheli/Desktop/Unine/MA1/Trustworthy/Trustworhty_AI_Watermarking/results/attacks/word_deletion/word_deletion_r0.4.jsonl
ratio 0.4: detected 429/600
Saved 600 records to /Users/paulmicheli/Desktop/Unine/MA1/Trustworthy/Trustworhty_AI_Watermarki

## 5. Summary

- Post-attack detection rate: share of the 600 attacked texts still detected.
- ASR: among the 583 texts detected before the attack, share no longer detected.
- Green fraction and scored tokens: mean change vs the original text.
95% Wilson confidence intervals for the rates.

In [6]:
from scipy.stats import binomtest

def wilson(k, n):
    ci = binomtest(k, n).proportion_ci(method="wilson")
    return f"[{ci.low:.1%}, {ci.high:.1%}]"

rows = []
print(f"{'ratio':>5} | {'detection rate':>24} | {'ASR (on 583)':>24} | "
      f"{'mean green':>10} | {'mean scored':>11} | {'words kept':>10}")
for ratio in RATIOS:
    recs = load_jsonl(OUT_DIR / f"word_deletion_r{ratio:.1f}.jsonl")
    n_det = sum(r["detected"] for r in recs)
    init = [r for r in recs if r["original_detected"]]
    n_removed = sum(not r["detected"] for r in init)
    green = np.mean([r["green_fraction"] for r in recs])
    scored = np.mean([r["n_scored"] for r in recs])
    kept = np.mean([r["n_words_attacked"] / r["n_words_original"] for r in recs])
    print(f"{ratio:5.1f} | {n_det/600:6.1%} {wilson(n_det, 600):>17} | "
          f"{n_removed/len(init):6.1%} {wilson(n_removed, len(init)):>17} | "
          f"{green:10.3f} | {scored:11.1f} | {kept:10.1%}")

ratio |           detection rate |             ASR (on 583) | mean green | mean scored | words kept
  0.0 |  97.5%    [95.9%, 98.5%] |   0.2%      [0.0%, 1.0%] |      0.448 |       184.0 |     100.0%
  0.1 |  94.8%    [92.8%, 96.3%] |   2.4%      [1.4%, 4.0%] |      0.430 |       173.0 |      90.0%
  0.2 |  91.2%    [88.6%, 93.2%] |   6.2%      [4.5%, 8.4%] |      0.412 |       159.8 |      80.0%
  0.3 |  87.0%    [84.1%, 89.5%] |  10.6%     [8.4%, 13.4%] |      0.396 |       144.4 |      70.0%
  0.4 |  71.5%    [67.8%, 75.0%] |  26.4%    [23.0%, 30.1%] |      0.379 |       127.1 |      60.0%
  0.5 |  53.0%    [49.0%, 57.0%] |  45.5%    [41.5%, 49.5%] |      0.363 |       108.4 |      50.0%
